# АО «Волковгеология» — Разведочный анализ данных
# Volkovgeology JSC — Exploratory Data Analysis

**Аналитик / Analyst:** Nurbol Sultanov  
**Дата / Date:** April 2020  

Первичный анализ операционных затрат на буровые работы по месторождениям урана (2018–2020).  
Initial exploration of drilling OPEX data across uranium deposits (2018–2020).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Загрузка данных / Load Data

In [2]:
operations = pd.read_csv('../data/raw/drilling_operations.csv', parse_dates=['дата_начала', 'дата_окончания'])
deposits = pd.read_csv('../data/raw/deposits.csv')
costs = pd.read_csv('../data/raw/cost_categories.csv')

print(f'Операции бурения: {operations.shape}')
print(f'Месторождения: {deposits.shape}')
print(f'Статьи затрат: {costs.shape}')

Операции бурения: (19419, 16)
Месторождения: (12, 8)
Статьи затрат: (9, 5)


In [3]:
print(f'Columns: {", ".join(operations.columns)}')
print()
print(f'Date range: {operations["дата_начала"].min().date()} to {operations["дата_начала"].max().date()}')
print(f'Unique deposits: {operations["код_месторождения"].nunique()}')
print(f'Unique wells: {operations["номер_скважины"].nunique()}')

Columns: номер_операции, код_месторождения, наименование_месторождения, номер_скважины, дата_начала, дата_окончания, глубина_план_м, глубина_факт_м, длительность_дней, код_статьи, план_тыс_тнг, факт_тыс_тнг, отклонение_тыс_тнг, год, месяц, квартал

Date range: 2018-01-01 to 2020-03-27
Unique deposits: 13
Unique wells: 2171


## 2. Качество данных / Data Quality

In [4]:
print('=== Пропуски / Missing Values ===')
print(f'Total nulls across all columns: {operations.isnull().sum().sum()}')

print('\n=== Дубликаты / Duplicates ===')
print(f'Duplicate operation IDs: {operations["номер_операции"].duplicated().sum()}')

print('\n=== Отрицательные суммы / Negative Amounts ===')
print(f'Negative факт_тыс_тнг: {(operations["факт_тыс_тнг"] < 0).sum()}')
print(f'Zero факт_тыс_тнг: {(operations["факт_тыс_тнг"] == 0).sum()}')

=== Пропуски / Missing Values ===
Total nulls across all columns: 0

=== Дубликаты / Duplicates ===
Duplicate operation IDs: 0

=== Отрицательные суммы / Negative Amounts ===
Negative факт_тыс_тнг: 0
Zero факт_тыс_тнг: 0


In [5]:
print('=== Ключевые показатели / Key Stats ===')
print(f'Всего записей: {len(operations):,}')
print(f'Уникальных скважин: {operations["номер_скважины"].nunique():,}')
print(f'Уникальных месторождений: {operations["код_месторождения"].nunique()}')
print(f'Суммарные затраты (факт): {operations["факт_тыс_тнг"].sum()/1000:,.1f} млн тнг')
print(f'Средняя стоимость записи: {operations["факт_тыс_тнг"].mean():,.1f} тыс тнг')

=== Ключевые показатели / Key Stats ===
Всего записей: 19,419
Уникальных скважин: 2,171
Уникальных месторождений: 13
Суммарные затраты (факт): 34,849.9 млн тнг
Средняя стоимость записи: 1,794.7 тыс тнг


## 3. Затраты по годам / OPEX by Year

In [6]:
yearly = operations.groupby('год').agg(
    факт_млн=('факт_тыс_тнг', lambda x: x.sum() / 1000),
    план_млн=('план_тыс_тнг', lambda x: x.sum() / 1000),
    скважин=('номер_скважины', 'nunique'),
    записей=('номер_операции', 'count')
).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total OPEX
x = yearly.index.astype(str)
width = 0.35
axes[0].bar(np.arange(len(x)) - width/2, yearly['план_млн'], width, label='План', color='#3498db')
axes[0].bar(np.arange(len(x)) + width/2, yearly['факт_млн'], width, label='Факт', color='#e74c3c')
axes[0].set_xticks(np.arange(len(x)))
axes[0].set_xticklabels(x)
axes[0].set_title('Затраты план/факт по годам (млн тнг)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('млн тнг')
axes[0].legend()

# Wells drilled
axes[1].bar(x, yearly['скважин'], color='#27ae60', edgecolor='white')
axes[1].set_title('Количество скважин по годам', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Скважин')

plt.suptitle('АО «Волковгеология» \u2014 Обзор затрат', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/yearly_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Затраты по месторождениям / OPEX by Deposit

In [7]:
by_deposit = operations.groupby('наименование_месторождения').agg(
    факт_млн=('факт_тыс_тнг', lambda x: x.sum() / 1000),
    скважин=('номер_скважины', 'nunique')
).sort_values('факт_млн', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if name == 'ТЕСТОВОЕ' else '#2c3e50' for name in by_deposit.index]
by_deposit['факт_млн'].plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Суммарные затраты по месторождениям (млн тнг)', fontsize=13, fontweight='bold')
ax.set_xlabel('млн тнг')
ax.set_ylabel('')

for i, v in enumerate(by_deposit['факт_млн']):
    ax.text(v + 20, i, f'{v:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/opex_by_deposit.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Затраты по статьям / OPEX by Cost Category

In [8]:
by_cost = operations.merge(costs[['код_статьи', 'наименование_статьи']], on='код_статьи', how='left')
cost_totals = by_cost.groupby('наименование_статьи')['факт_тыс_тнг'].sum().sort_values(ascending=True) / 1000

fig, ax = plt.subplots(figsize=(10, 6))
cost_totals.plot(kind='barh', ax=ax, color='#8e44ad', edgecolor='white')
ax.set_title('Затраты по статьям расходов (млн тнг)', fontsize=13, fontweight='bold')
ax.set_xlabel('млн тнг')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig('../reports/figures/opex_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Сезонность / Seasonality

In [9]:
monthly = operations.groupby(['год', 'месяц']).agg(
    факт_млн=('факт_тыс_тнг', lambda x: x.sum() / 1000),
    скважин=('номер_скважины', 'nunique')
).reset_index()

monthly['period'] = pd.to_datetime(monthly['год'].astype(str) + '-' + monthly['месяц'].astype(str).str.zfill(2) + '-01')

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(monthly['period'], monthly['факт_млн'], 'o-', color='#2c3e50', linewidth=1.5)
axes[0].set_title('Ежемесячные затраты (млн тнг)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('млн тнг')

axes[1].bar(monthly['period'], monthly['скважин'], color='#27ae60', width=20)
axes[1].set_title('Количество скважин в месяц', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Скважин')

plt.suptitle('Динамика затрат и бурения по месяцам', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/monthly_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Отклонения план/факт / Plan vs Actual Variance

In [10]:
operations['отклонение_пцт'] = (operations['факт_тыс_тнг'] - operations['план_тыс_тнг']) / operations['план_тыс_тнг'] * 100

print(f'Среднее отклонение: {operations["отклонение_тыс_тнг"].mean():+,.1f} тыс тнг (факт выше плана)')
print(f'Медиана отклонения: {operations["отклонение_тыс_тнг"].median():+,.1f} тыс тнг')
print(f'Записей с перерасходом >15%: {(operations["отклонение_пцт"] > 15).sum()} ({(operations["отклонение_пцт"] > 15).mean():.1%})')
print(f'Записей с экономией >20%: {(operations["отклонение_пцт"] < -20).sum()} ({(operations["отклонение_пцт"] < -20).mean():.1%})')

Среднее отклонение: +127.3 тыс тнг (факт выше плана)
Медиана отклонения: +62.1 тыс тнг
Записей с перерасходом >15%: 1,793 (9.2%)
Записей с экономией >20%: 1,847 (9.5%)


In [11]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(operations['отклонение_пцт'].clip(-30, 30), bins=60, color='#3498db', edgecolor='white', alpha=0.8)
ax.axvline(x=0, color='black', linewidth=1)
ax.axvline(x=15, color='red', linestyle='--', alpha=0.7, label='Порог перерасхода (+15%)')
ax.set_title('Распределение отклонений факт/план (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Отклонение (%)')
ax.set_ylabel('Количество записей')
ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/variance_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Стоимость бурения на метр / Cost per Meter

In [12]:
# Aggregate cost per well, then cost per meter
well_costs = operations.groupby(['номер_скважины', 'наименование_месторождения', 'глубина_факт_м']).agg(
    total_cost=('факт_тыс_тнг', 'sum')
).reset_index()

well_costs['cost_per_meter'] = well_costs['total_cost'] / well_costs['глубина_факт_м']

# Box plot by deposit
fig, ax = plt.subplots(figsize=(14, 6))
deposit_order = well_costs.groupby('наименование_месторождения')['cost_per_meter'].median().sort_values().index
well_costs['наименование_месторождения'] = pd.Categorical(
    well_costs['наименование_месторождения'], categories=deposit_order, ordered=True
)

sns.boxplot(data=well_costs, x='наименование_месторождения', y='cost_per_meter', ax=ax,
            palette='Set2', fliersize=3)
ax.set_title('Стоимость бурения на метр по месторождениям (тыс тнг/м)', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('тыс тнг / м')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../reports/figures/cost_per_meter.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Предварительные выводы / Initial Findings

### RU:
- Данные покрывают 27 месяцев (янв 2018 \u2014 март 2020), ~19.4K записей по 2,171 скважине
- Суммарные факт. затраты: ~34.8 млрд тнг
- Выраженная сезонность: зимние месяцы на 20-25% дороже летних (ГСМ, логистика)
- ~9% записей имеют аномальный перерасход (>15% от плана) \u2014 требуют отдельного разбора
- Заметное снижение активности в Q1 2020 (COVID-19)
- Стоимость метра бурения варьируется в 2-3x между месторождениями

### EN:
- Data covers 27 months (Jan 2018 \u2014 Mar 2020), ~19.4K records across 2,171 wells
- Total actual OPEX: ~34.8B KZT
- Clear seasonality: winter months 20-25% more expensive (fuel, logistics)
- ~9% of records have anomalous overruns (>15% over plan) \u2014 need separate review
- Notable activity decline in Q1 2020 (COVID-19)
- Cost per meter varies 2-3x between deposits

**Следующий шаг / Next:** детальный анализ аномалий и стоимости по статьям.